## **Project 2 - Alex Eyre**
### In vivo CRISPR screens identify the E3 ligase Cop1 as a modulator of macrophage infiltration and cancer immunotherapy target
### (Wang et al., Cell 2021)
=============================================================================

In [1]:
# =============================================================================
# Import libraries.
# =============================================================================

import os
import sys
import logging
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
from pathlib import Path

import mageck
#sys.path.insert(0, "/mnt/c/Users/aweyr/OneDrive/Documents/Programs/DrugZ")
#import drugz as dz

In [2]:
# =============================================================================
# Startup R.
# =============================================================================

logging.getLogger("rpy2").setLevel(logging.CRITICAL)
%load_ext rpy2.ipython

In [3]:
#%%R
# =============================================================================
# Load R Packages.
# =============================================================================

#suppressPackageStartupMessages({library("DESeq2") # DESeq2: 1.50.2
#                               library("clusterProfiler") # clusterProfiler: 4.18.4
#                               library("ComplexHeatmap") # ComplexHeatmap: 2.26.1
#                               library("enrichplot") # enrichplot: 1.30.5 
#                               library("org.Mm.eg.db") # org.Mm.eg.db: 1.30.5 
#                               library("CellChat")}) # CellChat

=============================================================================
#### Functions
=============================================================================

In [4]:
def parse_mageck_results(
    df: pd.DataFrame,
    *,
    header: bool = True,
    columns_per_comparison: int = 14,
    reset_index: bool = True,
) -> dict[str, pd.DataFrame]:
    
    """
    Parse one or more MAGeCK comparison results from a results table.

    Parameters
    ----------
    df : pd.DataFrame
        Raw MAGeCK results table read with header=None.

    header : bool
        Whether the table contains an additional first row with comparison
        names.

        If True, row 0 contains comparison names and row 1 contains the
        MAGeCK output headers.

        If False, row 0 contains the MAGeCK output headers.

    columns_per_comparison : int
        Number of columns in each MAGeCK comparison block.

    reset_index : bool
        Whether to reset the index after removing metadata/header rows.

    Returns
    -------
    dict[str, pd.DataFrame]
        Dictionary where keys are comparison names and values are cleaned
        MAGeCK result tables.
    """

    mageck_results = {}

    n_comparisons = df.shape[1] // columns_per_comparison

    for comparison_idx in range(n_comparisons):
        start_idx = (comparison_idx * columns_per_comparison) + comparison_idx
        end_idx = start_idx + columns_per_comparison

        subset = df.iloc[:, list(range(start_idx, end_idx))].copy()
        
        if header:
            comparison_name = str(subset.columns[0]).strip() 
            subset.columns = subset.iloc[0]
            subset = subset.iloc[1:]
        else:
            comparison_name = f"comparison_{comparison_idx + 1}"

        # Clean column names
        subset.columns = subset.columns.astype(str).str.strip()

        # Fix possible malformed id column
        if subset.columns[0] != "id":
            subset = subset.rename(columns={subset.columns[0]: "id"})

        # Convert MAGeCK statistic columns to numeric
        for col in subset.columns:
            if col != "id":
                subset[col] = pd.to_numeric(subset[col], errors="coerce")

        if reset_index:
            subset = subset.reset_index(drop = True)

        mageck_results[comparison_name] = subset

    return mageck_results

=============================================================================
#### In Vivo screens with the MusCK library uncover classic and novel regulators of immune evasion.
=============================================================================

In [5]:
# =============================================================================
# Define project paths.
# =============================================================================
project_dir = Path.cwd().parents[0]
data_dir = project_dir / "data" / "supplemental"

In [6]:
# =============================================================================
# Load sgRNA Library Design.
# =============================================================================
MusCK_library_design = pd.read_csv(data_dir / "Table_S1_MusCK_library_design.txt", sep = "\t")

#### MusCK Description
```
sgID: unique identifiers for each sgRNA
seq: sgRNA sequence
geneID: non-unique gene identifier, each gene contains 5 sgRNAs
```

In [7]:
# =============================================================================
# Parse in published supplemental tables.
# =============================================================================
# Experiment 1 - In Vitro
MusCK_S2_invitro = pd.read_csv(data_dir / "Table_S2_In_vitro_MusCK_library_screening_on_4T1_cells.txt", sep = "\t", header = 0)
MusCK_invitro = parse_mageck_results(MusCK_S2_invitro, header = False)

# Experiment 1 - In Vivo
MusCK_S3_invivo = pd.read_csv(data_dir / "Table_S3_In_vivo_MusCK_library_screening_on_4T1_cells.txt", sep = "\t", header = 0)
MusCK_invivo = parse_mageck_results(MusCK_S3_invivo, header = True)

In [8]:
# =============================================================================
# In Vitro Data.
# =============================================================================
MusCK_invitro['comparison_1'].describe()

,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4721.000000,4.721000e+03,4721.000000,4721.000000,4721.000000,4721.000000,4721.000000,4.721000e+03,4721.000000,4721.000000,4721.000000,4721.000000,4721.000000
mean,4.998729,5.404764e-01,0.609285,0.853753,2418.469180,1.777801,-0.216388,4.249298e-01,0.516839,0.903303,2371.393984,1.900445,-0.216388
std,0.035631,3.539321e-01,0.333280,0.325152,1369.323746,1.546724,0.988153,3.620037e-01,0.332265,0.156922,1372.481899,1.354622,0.988153
min,4.000000,6.390000e-09,0.000001,0.000108,17.000000,0.000000,-7.786600,7.900000e-13,0.000001,0.000825,1.000000,0.000000,-7.786600
25%,5.000000,2.009100e-01,0.377180,0.999999,1234.000000,1.000000,-0.322900,8.487900e-02,0.208440,0.843446,1183.000000,1.000000,-0.322900
50%,5.000000,5.793500e-01,0.719240,0.999999,2420.000000,1.000000,0.066535,3.067300e-01,0.500070,0.999999,2367.000000,2.000000,0.066535
75%,5.000000,8.832700e-01,0.885580,0.999999,3604.000000,3.000000,0.341160,8.045000e-01,0.831230,0.999999,3559.000000,3.000000,0.341160
max,5.000000,1.000000e+00,1.000000,0.999999,4787.000000,5.000000,2.666900,1.000000e+00,1.000000,0.999999,4774.000000,5.000000,2.666900


In [9]:
# =============================================================================
# In Vivo Data - BALB/c VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK_invivo['BALB/c VS. BALB/c Foxn1nu/nu'].describe()

,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000
mean,4.833543,0.363209,0.496808,0.912950,2388.946817,2.418760,-1.386158,0.520180,0.520184,0.999940,2388.872487,0.048367,-1.385767
std,4.761333,0.312742,0.308948,0.124265,1379.284955,2.318353,0.803412,0.298925,0.298972,0.001037,1379.261691,0.229649,0.803348
min,1.000000,0.000006,0.000028,0.133663,1.000000,0.000000,-4.942500,0.000108,0.000281,0.964109,1.000000,0.000000,-4.942500
25%,5.000000,0.084332,0.216638,0.861753,1194.750000,2.000000,-1.926525,0.261557,0.260980,0.999970,1194.750000,0.000000,-1.926000
50%,5.000000,0.280525,0.502775,0.999773,2388.500000,2.000000,-1.335600,0.517635,0.518095,0.999970,2388.500000,0.000000,-1.335600
75%,5.000000,0.613215,0.778020,0.999773,3583.250000,3.000000,-0.750020,0.783573,0.783382,0.999970,3583.250000,0.000000,-0.749630
max,273.000000,0.999780,0.999810,0.999810,4777.000000,117.000000,1.037000,0.999960,0.999970,0.999970,4777.000000,5.000000,1.037000


In [10]:
# =============================================================================
# In Vivo Data - BALB/c +OVA VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK_invivo['BALB/c +OVA VS. BALB/c Foxn1nu/nu'].describe()

,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000,4776.000000
mean,4.844430,0.367504,0.503720,0.927609,2388.500000,2.424623,-0.866356,0.517782,0.517773,0.999667,2388.500000,0.000209,-0.866147
std,4.767108,0.310481,0.305987,0.131204,1378.856773,2.486608,0.796412,0.296588,0.296673,0.005567,1378.856773,0.014470,0.796354
min,1.000000,0.000043,0.000096,0.392047,1.000000,0.000000,-4.514800,0.000108,0.000146,0.698020,1.000000,0.000000,-4.514800
25%,5.000000,0.088826,0.223285,0.891473,1194.750000,2.000000,-1.329550,0.259217,0.258765,0.999933,1194.750000,0.000000,-1.328550
50%,5.000000,0.290795,0.513685,0.999814,2388.500000,2.000000,-0.762400,0.520430,0.520050,0.999933,2388.500000,0.000000,-0.761450
75%,5.000000,0.612847,0.776998,0.999814,3582.250000,3.000000,-0.405190,0.777005,0.776565,0.999933,3582.250000,0.000000,-0.405190
max,273.000000,0.999790,0.999810,0.999814,4776.000000,130.000000,2.091900,0.999920,0.999930,0.999933,4776.000000,1.000000,2.091900


#### MusCK Statistics Description

```
id: geneID
num: number of sgRNAs representing the gene
neg|score: gene-level p-value for negative selection (depletion), smaller values indicate stronger evidence that disruption of this gene decreases cell fitness under the tested condition
neg|p-value: one-sided p-value testing whether sgRNAs targeting the gene are significantly depleted relative to the background distribution
neg|fdr: Benjamini-Hochberg false discovery rate (FDR) adjusted p-value for negative selection. Controls for multiple hypothesis testing
neg|rank: rank of the gene among all genes for negative selection (1 = strongest depletion)
neg|goodsgrna: number of sgRNAs for the gene whose log-fold changes contributed to the negative-selection score after DrugZ filtering and directional consistency checks
neg|lfc: mean log₂ fold change (LFC) of sgRNAs targeting the gene, more negative values indicate stronger depletion
pos|score: gene-level p-value for positive selection (enrichment),s maller values indicate stronger evidence that disruption of this gene decreases cell fitness under the tested condition
pos|p-value:  one-sided p-value testing whether sgRNAs targeting the gene are significantly enriched relative to the background distribution
pos|fdr: Benjamini-Hochberg false discovery rate (FDR) adjusted p-value for positive selection
pos|rank: rank of the gene among all genes for positive selection (1 = strongest enrichment)
pos|goodsgrna: number of sgRNAs contributing to the positive-selection score after DrugZ filtering and directional consistency checks
pos|lfc: mean log₂ fold change (LFC) of sgRNAs targeting the gene. More positive values indicate stronger enrichment
```

=============================================================================
#### Second-round MusCK screens identify Cop1 as a regulator of TNBC progression
=============================================================================

In [11]:
# =============================================================================
# Parse in published supplemental tables
# =============================================================================
# Experiment 2 - In Vivo
MusCK2_S4_invivo = pd.read_csv(data_dir / "Table_S4_In_vivo_MusCK2_0_library_screening_on_cancer_cells.txt", sep = "\t", header = 0)
MusCK2_invivo = parse_mageck_results(MusCK2_S4_invivo, header = True)

In [12]:
# =============================================================================
# In Vivo Data - BALB/c VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK2_invivo['BALB/c VS. BALB/c Foxn1nu/nu'].describe()

,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83.000000,8.300000e+01,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000
mean,9.626506,5.387316e-01,0.605249,0.815756,42.000000,2.277108,-0.089298,0.530494,0.543996,0.967122,42.000000,0.590361,-0.089298
std,7.359598,3.638219e-01,0.349769,0.369335,24.103942,2.275604,0.672821,0.332218,0.318906,0.113022,24.103942,1.012557,0.672821
min,7.000000,1.460000e-12,0.000005,0.000138,1.000000,0.000000,-2.763400,0.000742,0.002922,0.242537,1.000000,0.000000,-2.763400
25%,8.000000,2.233200e-01,0.400435,0.999905,21.500000,1.000000,-0.209090,0.234815,0.274200,0.982822,21.500000,0.000000,-0.209090
50%,8.000000,5.528600e-01,0.689990,0.999905,42.000000,2.000000,0.077655,0.557900,0.558810,0.999995,42.000000,0.000000,0.077655
75%,8.000000,8.875600e-01,0.892500,0.999905,62.500000,3.000000,0.258925,0.816075,0.828140,0.999995,62.500000,1.000000,0.258925
max,47.000000,9.999300e-01,0.999910,0.999905,83.000000,8.000000,0.932680,1.000000,1.000000,0.999995,83.000000,6.000000,0.932680


In [13]:
# =============================================================================
# In Vivo Data - BALB/c +OVA VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK2_invivo['BALB/c +OVA VS. BALB/c Foxn1nu/nu'].describe()

,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83.000000,8.300000e+01,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000
mean,9.626506,5.397791e-01,0.597983,0.833356,42.000000,2.072289,-0.070423,0.547568,0.562198,0.909344,42.000000,0.578313,-0.070423
std,7.359598,3.497288e-01,0.336173,0.336344,24.103942,2.240335,0.688966,0.342873,0.331584,0.188848,24.103942,0.938609,0.688966
min,7.000000,8.260000e-14,0.000005,0.000138,1.000000,0.000000,-3.315900,0.000003,0.000015,0.001244,1.000000,0.000000,-3.315900
25%,8.000000,2.078400e-01,0.376400,0.999585,21.500000,0.000000,-0.272945,0.212410,0.229990,0.887759,21.500000,0.000000,-0.272945
50%,8.000000,6.305100e-01,0.715700,0.999585,42.000000,1.000000,0.059533,0.613420,0.614110,0.999995,42.000000,0.000000,0.059533
75%,8.000000,8.642400e-01,0.864660,0.999585,62.500000,3.000000,0.326865,0.855265,0.855680,0.999995,62.500000,1.000000,0.326865
max,47.000000,9.996500e-01,0.999590,0.999585,83.000000,8.000000,1.253600,1.000000,1.000000,0.999995,83.000000,5.000000,1.253600


In [14]:
# =============================================================================
# In Vivo Data - BALB/c +OVA+anti-PD-1 antibody VS. BALB/c Foxn1nu/nu.
# =============================================================================
MusCK2_invivo['BALB/c +OVA+anti-PD-1 antibody VS. BALB/c Foxn1nu/nu'].describe()

,num,neg|score,neg|p-value,neg|fdr,neg|rank,neg|goodsgrna,neg|lfc,pos|score,pos|p-value,pos|fdr,pos|rank,pos|goodsgrna,pos|lfc
count,83.000000,8.300000e+01,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000
mean,9.626506,4.589721e-01,0.565672,0.842935,42.000000,2.843373,-0.094745,0.517027,0.555628,0.938011,42.000000,1.180723,-0.094745
std,7.359598,3.572968e-01,0.342935,0.308555,24.103942,2.456857,0.728216,0.344296,0.318018,0.174085,24.103942,1.499143,0.728216
min,7.000000,2.010000e-14,0.000005,0.000415,1.000000,0.000000,-3.625400,0.000029,0.000145,0.012023,1.000000,0.000000,-3.625400
25%,8.000000,9.200950e-02,0.234410,0.889435,21.500000,1.000000,-0.458130,0.181900,0.303670,0.999995,21.500000,0.000000,-0.458130
50%,8.000000,4.498100e-01,0.656630,0.997178,42.000000,2.000000,0.043350,0.552980,0.570350,0.999995,42.000000,1.000000,0.043350
75%,8.000000,8.207200e-01,0.872470,0.997178,62.500000,4.000000,0.329415,0.832645,0.832610,0.999995,62.500000,2.000000,0.329415
max,47.000000,9.948500e-01,0.997180,0.997178,83.000000,12.000000,1.685600,1.000000,1.000000,0.999995,83.000000,8.000000,1.685600


=============================================================================
#### Gene Annotation
=============================================================================